# Feature engineering et split temporel

Cette étape prépare la série nationale pour la modélisation day-ahead. Les variables utilisent uniquement le calendrier et les observations passées. Aucun `shuffle=True` n'est utilisé.

In [ ]:
from pathlib import Path
import pandas as pd

# Le notebook peut être lancé depuis notebooks/ ou depuis la racine du projet.
DATA_PATH = Path('../data/eco2mix.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/eco2mix.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable : {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').drop_duplicates('timestamp').reset_index(drop=True)

print('Fichier utilisé :', DATA_PATH.resolve())
print('Dimensions :', df.shape)
display(df.head())
print(df.dtypes)

Dimensions : (236563, 2)


,timestamp,consommation_mw
0,2012-12-31 23:30:00+00:00,59662.0
1,2013-01-01 00:00:00+00:00,57866.0
2,2013-01-01 00:30:00+00:00,57747.0
3,2013-01-01 01:00:00+00:00,57234.0
4,2013-01-01 01:30:00+00:00,56649.0


timestamp          datetime64[ns, UTC]
consommation_mw                float64
dtype: object


## 1. Régulariser la fréquence

La donnée est demi-horaire. On remet la série sur une grille de 30 minutes avant de calculer les lags. Les timestamps absents restent manquants : on ne crée pas artificiellement une consommation future.

In [3]:
df = (df.set_index('timestamp')
        .asfreq('30min')
        .rename_axis('timestamp')
        .reset_index())

print('Valeurs manquantes :')
display(df.isna().sum().to_frame('nulls'))
print('Pas temporel :', df['timestamp'].diff().value_counts().head())

Valeurs manquantes :


,nulls
timestamp,0
consommation_mw,26


Pas temporel : timestamp
0 days 00:30:00    236588
Name: count, dtype: int64


## 2. Créer les features sans fuite

`lag_48` correspond à la consommation 24 heures auparavant et `lag_336` à la consommation 7 jours auparavant. Les variables calendaires sont connues au moment de la prévision. Les lignes sans historique complet sont supprimées après création des features.

In [4]:
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['dayofweek'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['dayofyear'] = df['timestamp'].dt.dayofyear

# Les lags regardent uniquement le passé.
df['lag_1'] = df['consommation_mw'].shift(1)
df['lag_48'] = df['consommation_mw'].shift(48)
df['lag_96'] = df['consommation_mw'].shift(96)
df['lag_336'] = df['consommation_mw'].shift(336)

feature_columns = [
    'hour', 'minute', 'dayofweek', 'month', 'dayofyear',
    'lag_1', 'lag_48', 'lag_96', 'lag_336'
]
target_column = 'consommation_mw'
model_df = df.dropna(subset=feature_columns + [target_column]).copy()

print('Dimensions après création des features :', model_df.shape)
display(model_df[['timestamp'] + feature_columns + [target_column]].head())

Dimensions après création des features : (236136, 11)


,timestamp,hour,minute,dayofweek,month,dayofyear,lag_1,lag_48,lag_96,lag_336,consommation_mw
336,2013-01-07 23:30:00+00:00,23,30,0,1,7,67629.0,60756.0,61035.0,59662.0,65869.0
337,2013-01-08 00:00:00+00:00,0,0,1,1,8,65869.0,58146.0,58174.0,57866.0,62941.0
338,2013-01-08 00:30:00+00:00,0,30,1,1,8,62941.0,58055.0,57992.0,57747.0,62831.0
339,2013-01-08 01:00:00+00:00,1,0,1,1,8,62831.0,57641.0,57159.0,57234.0,62279.0
340,2013-01-08 01:30:00+00:00,1,30,1,1,8,62279.0,57515.0,56578.0,56649.0,62065.0


## 3. Découpage temporel train/test

Le test contient les trois derniers mois. Le modèle apprend uniquement sur les dates antérieures. Ce choix reproduit le cas réel : prévoir le futur à partir du passé.

In [5]:
test_start = model_df['timestamp'].max() - pd.DateOffset(months=3)
train = model_df[model_df['timestamp'] < test_start].copy()
test = model_df[model_df['timestamp'] >= test_start].copy()

X_train = train[feature_columns]
y_train = train[target_column]
X_test = test[feature_columns]
y_test = test[target_column]

print('Début du test :', test_start)
print('Train :', X_train.shape, '|', train['timestamp'].min(), '->', train['timestamp'].max())
print('Test  :', X_test.shape, '|', test['timestamp'].min(), '->', test['timestamp'].max())
assert train['timestamp'].max() < test['timestamp'].min()
print('Découpage temporel valide : aucune date du train ne dépasse le début du test.')

Début du test : 2026-03-30 21:30:00+00:00
Train : (231719, 9) | 2013-01-07 23:30:00+00:00 -> 2026-03-30 21:00:00+00:00
Test  : (4417, 9) | 2026-03-30 21:30:00+00:00 -> 2026-06-30 21:30:00+00:00
Découpage temporel valide : aucune date du train ne dépasse le début du test.


## Conclusion

Les données sont prêtes pour comparer une baseline de persistance (`lag_48`) et un modèle LightGBM. La prochaine étape sera d'entraîner les trois runs MLflow demandés, en calculant RMSE, MAE et R2 uniquement sur `test`.